# BBC News: Entertainment  Category Sub-classification
## NLP Pipeline: BGE-M3 Embeddings + BERTopic (UMAP + HDBSCAN + c-TF-IDF)

**Architecture:**
- Layer 1: Data Ingestion — Raw .txt files → List[String]
- Layer 2: BGE-M3 Embeddings — 8192 token context, 1024 dims
- Layer 3: UMAP — Non-linear reduction 1024 → 5 dims
- Layer 4: HDBSCAN — Automatic density-based clustering
- Layer 5a: CountVectorizer — Removes noise words per cluster
- Layer 5b: c-TF-IDF — Distinctive keywords across clusters
- Layer 6: KeyBERTInspired — Semantic keyword refinement
- Layer 7: Output — DataFrame [filename, category, sub_category, confidence_score, preview]

**Dataset:** BBC News Entertainment Category — 511 raw articles
**Final Output:** DataFrame mapping every article to its Entertainment sub-category

## Environment Setup & Imports
### Loading all required libraries for the full BERTopic pipeline

In [ ]:
# Core libraries
import os
import re
import pickle
import numpy as np
import pandas as pd
from loader import load_data

# Sentence Transformers — BGE-M3
from sentence_transformers import SentenceTransformer

# BERTopic pipeline
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

print("All imports successful")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

## Data Ingestion Layer
### Loading raw Entertainment articles from local storage
### Source: BBC News dataset for business category
### No aggressive cleaning at this stage, BGE-M3 needs full sentence structure

In [ ]:
import os

# Load all articles
load_data = load_data('../data/entertainment')

# Load all txt files
Entertainment_docs = []
Entertainment_files = []

for filename in sorted(os.listdir(load_data)):
    if filename.endswith('.txt'):
        file_path = os.path.join(load_data, filename)
        with open(file_path, encoding='utf-8', errors='ignore') as f:
            text = f.read()
        Entertainment_docs.append(text)
        Entertainment_files.append(filename)

print(f"Total Entertainment articles loaded: {len(Entertainment_docs)}")
print(f"\nSample filename: {Entertainment_files[0]}")
print(f"\nSample article preview:")
print(f"{Entertainment_docs[0][:300]}")

## Data Normalization Layer
### Light cleaning only preserving full sentence structure for BGE-M3
### We only remove: encoding errors, excessive whitespace, HTML tags
### Punctuation, proper nouns, stopwords intentionally preserved

In [ ]:
def light_clean(text):
    # Fix encoding errors
    text = text.encode('utf-8', 'ignore').decode('utf-8')

    # Remove HTML tags if present
    text = re.sub(r'<.*?>', '', text)

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply light cleaning
cleaned_Entertainment = [light_clean(doc) for doc in Entertainment_docs]

print(f"Cleaning complete — {len(cleaned_Entertainment)} articles processed")
print(f"\nBefore: {Entertainment_docs[0][:200]}")
print(f"\nAfter: {cleaned_Entertainment[0][:200]}")

##  Deduplication
### Removing duplicate articles before embedding generation
### Identical documents produce identical vectors hurts clustering quality
### Preserving filename mapping for final DataFrame output

In [ ]:
# Remove duplicates while preserving filename mapping
seen = set()
unique_Entertainment = []
unique_files = []

for doc, filename in zip(cleaned_Entertainment, Entertainment_files):
    if doc not in seen:
        seen.add(doc)
        unique_Entertainment.append(doc)
        unique_files.append(filename)

print(f"Before deduplication: {len(cleaned_Entertainment)} articles")
print(f"After deduplication:  {len(unique_Entertainment)} articles")
print(f"Duplicates removed:   {len(cleaned_Entertainment) - len(unique_Entertainment)}")

## Embedding Layer (BGE-M3)
### Model: BAAI/bge-m3 —> 8192 token context window, 1024 dimensional output
### Full-sequence self-attention transformer from BAAI
### Every article processed as one complete unit, no truncation
### Produces dense semantic vectors capturing meaning, context and intent

In [ ]:
# Load BGE-M3 model
print("Loading BGE-M3 model...")
model = SentenceTransformer('BAAI/bge-m3')
print("BGE-M3 loaded successfully")
print(f"Max sequence length: {model.max_seq_length}")

##  Generate BGE-M3 Document Embeddings
### Each article is encoded as a single 1024-dimensional semantic vector
### BGE-M3 reads the entire document using full-sequence self-attention
### batch_size=32 processes 32 articles at a time to manage memory efficiently
### show_progress_bar=True lets us track encoding progress

In [ ]:
print("Generating BGE-M3 embeddings for business articles...")
print("This will take a few minutes...")

embeddings = model.encode(
    unique_Entertainment,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"\nEmbeddings generated successfully")
print(f"Shape: {embeddings.shape}")
print(f"Each article \u2192 {embeddings.shape[1]} dimensional vector")

## Fit BERTopic to BGE-M3 Embeddings
### Passing pre-computed BGE-M3 embeddings directly into BERTopic
### BERTopic runs all 6 modules sequentially:
### UMAP → HDBSCAN → CountVectorizer → c-TF-IDF → KeyBERTInspired
### topics_ contains sub-category assignment for every article
### probs_ contains confidence score for each assignment

In [ ]:
# Fit BERTopic using pre-computed BGE-M3 embeddings
print("Running BERTopic pipeline...")
print("UMAP \u2192 HDBSCAN \u2192 Vectorizer \u2192 c-TF-IDF \u2192 KeyBERT")
print("This may take a few minutes...")

topics, probs = topic_model.fit_transform(
    unique_Entertainment,
    embeddings=embeddings
)

print(f"\nBERTopic complete")
print(f"Total articles processed: {len(topics)}")
print(f"Sub-categories discovered: {len(set(topics)) - 1}")
print(f"Noise articles (topic -1): {topics.count(-1)}")

### Storage: Final DataFrame

Now, let's create the final DataFrame as specified in your architecture. It will include the original filename, the main category ('Business'), the assigned sub-category (from BERTopic), and a preview of the article content.

In [ ]:
final_df = pd.DataFrame({
    'filename': unique_files,
    'category': 'Entertainment ',
    'sub_category_id': topics,
    'preview': [doc[:300] + '...' for doc in unique_Entertainment]
})

# Get human-readable topic names for sub-categories
topic_names = topic_model.get_topic_info()['Name']
# Create a mapping from topic ID to topic name
topic_id_to_name = dict(zip(topic_model.get_topic_info().Topic, topic_model.get_topic_info().Name))

# Map sub_category_id to sub_category_name
final_df['sub_category_name'] = final_df['sub_category_id'].map(topic_id_to_name)

# Handle noise articles (-1 topic_id) as 'Noise' or 'Unassigned'
final_df['sub_category_name'] = final_df['sub_category_name'].fillna('Noise')

# Reorder and select final columns
final_df = final_df[['filename', 'category', 'sub_category_id', 'sub_category_name', 'preview']]

print(f"Final DataFrame created with {len(final_df)} entries.")
display(final_df)

### Final DataFrame with Huamn labels

The final DataFrame. This will include the original filename, the main category ('Entertainment '), the assigned sub-category (human labelling ), and a preview of the article content.

In [ ]:
# FINAL DATAFRAME WITH CONFIDENCE SCORES & NOISE RELABELLING

# Human-readable label mapping
topic_labels = {
    0:  "Music Artists",
    1:  "Film Awards",
    2:  "Television Entertainment",
    3:  "Box Office Charts",
    4:  "Theatre Musicals",
    5:  "Film Festivals",
    6:  "Franchise Films",
    7:  "Music Industry",
    # newly labelled (formerly noise + splits) 
    8:  "Books Literature",       # book prizes, authors, literary awards
    9:  "Celebrity Culture",      # actors, paparazzi, reality TV, obituaries
    10: "Charity Music",          # Band Aid, tsunami concerts, benefit gigs
    11: "Film Industry",          # production stats, diversity, women in film
    -1: "Unassigned"
}

# Manual remap: misclassified articles inside existing topics
# Topic 0 non-music articles → correct labels
manual_remap = {
    # Art gallery / visual art → Celebrity Culture
    "001.txt": 9,    # Tate Britain interactive Christmas tree
    "018.txt": 9,    # Versace art portfolio sale
    "020.txt": 9,    # Reynolds portrait public showing

    # Theatre → Theatre Musicals
    "010.txt": 4,    # Uganda bans Vagina Monologues

    # Books/Authors → Books Literature
    "016.txt": 8,    # Arthur Hailey novelist obituary

    # Actor lawsuits / celebrity → Celebrity Culture
    "050.txt": 9,    # Bruce Willis suing Revolution Studios
    "110.txt": 9,    # Suge Knight arrested
    "135.txt": 9,    # Vibe awards stabbing
    "169.txt": 9,    # Robert Blake gun residue
    "282.txt": 9,    # 50 Cent G-Unit feud
    "288.txt": 9,    # 50 Cent feud ends
    "327.txt": 9,    # Nicole Kidman paparazzi

    # Film → Film Industry
    "378.txt": 11,   # Debra Hill Halloween screenwriter dies

    # Glastonbury → Music Artists (stays in 0, it's a music festival)
    # "137.txt" stays in 0 — Glastonbury ID cards is music

    # Topic 2 non-TV articles → correct labels
    "006.txt": 4,    # Bennett History Boys → Theatre Musicals
    "008.txt": 4,    # West End Evening Standard Awards → Theatre Musicals
    "108.txt": 9,    # Michael Jackson trial → Celebrity Culture
    "113.txt": 9,    # Wal-Mart sued over lyrics → Celebrity Culture
    "187.txt": 2,    # Big Brother eviction → stays TV (has TV signals)
    "205.txt": 9,    # Wife Swap copycat lawsuit → Celebrity Culture
    "225.txt": 2,    # Big Brother Jackie Stallone → stays TV
    "232.txt": 4,    # Harry Connick Jr Broadway → Theatre Musicals
    "275.txt": 4,    # The Producers West End awards → Theatre Musicals

    # Topic 7 Bollywood articles → Film Industry
    "349.txt": 11,   # India deports Bollywood actress
    "366.txt": 11,   # Bollywood DVD fraudster jailed
}

# Noise remap: all 51 noise articles
noise_remap = {
    # Books Literature (8)
    "004.txt": 8,    # Richard and Judy book award
    "007.txt": 8,    # Levy tipped for Whitbread prize
    "013.txt": 8,    # Levy takes Whitbread novel prize
    "014.txt": 8,    # Adventure tale tops awards — children's book
    "015.txt": 8,    # Mutant book wins Guardian prize
    "017.txt": 8,    # Spark heads world Booker list
    "024.txt": 8,    # Paraguay novel wins US book prize
    "027.txt": 8,    # Lit Idol search for author

    # Celebrity Culture (9)
    "019.txt": 9,    # Christian Slater Broadway Glass Menagerie
    "036.txt": 9,    # Tom Hanks Polar Express premiere
    "043.txt": 9,    # Stars pay tribute to Ossie Davis
    "044.txt": 9,    # Ossie Davis found dead
    "045.txt": 9,    # Gerard Jugnot best-paid French actor
    "089.txt": 9,    # Oscar nominee Dan O'Herlihy dies
    "176.txt": 9,    # Lisa Kudrow new sitcom
    "181.txt": 9,    # Chris Evans selling possessions
    "190.txt": 9,    # Cyril Fletcher comedian dies
    "198.txt": 9,    # Brookside actress Keaveney dies
    "307.txt": 9,    # Carry On star Patsy Rowlands dies
    "311.txt": 9,    # Ronnie Corbett attacks dumbed-down TV

    # Charity Music (10)
    "002.txt": 10,   # Jarre Copenhagen Andersen concert
    "104.txt": 10,   # Glasgow tsunami benefit gig
    "109.txt": 10,   # Band Aid retains number one
    "111.txt": 10,   # Elton John Paris tsunami concert
    "130.txt": 10,   # Cliff Richard charity single
    "133.txt": 10,   # McCartney Super Bowl performance
    "143.txt": 10,   # Blair buys Band Aid copies
    "157.txt": 10,   # We Are The World re-released
    "159.txt": 10,   # Band Aid 20 storms to No 1
    "160.txt": 10,   # iTunes selling Band Aid song

    # Film Industry (11)
    "054.txt": 11,   # Mumbai bombs movie postponed
    "063.txt": 11,   # Tautou film Cesar nominations
    "067.txt": 11,   # Low-budget film wins Cesar
    "073.txt": 11,   # Ray DVD beats box office
    "085.txt": 11,   # Bollywood priest affair film
    "194.txt": 11,   # Ethnic producers face barriers
    "202.txt": 11,   # BBC licence fee scrutiny
    "303.txt": 11,   # Film production falls 40% in UK
    "351.txt": 11,   # Women in film earning less
    "362.txt": 11,   # Original Exorcist to be screened
    "381.txt": 11,   # Lost Doors frontman movie found

    # Television Entertainment (2)
    "214.txt": 2,    # Supervolcano BBC drama
    "220.txt": 2,    # Bafta Interactive Awards
    "313.txt": 2,    # BBC invests £9m in new comedy

    # Theatre Musicals (4)
    "023.txt": 4,    # Conductor Viotti dies — opera
    "342.txt": 4,    # US musicologist recreates Bach score

    # Music Artists (0)
    "118.txt": 0,    # REM concerts cancelled
    "136.txt": 0,    # Christmas song formula
    "228.txt": 0,    # Joy Division film announcement
    "269.txt": 0,    # George Michael retirement film
    "280.txt": 0,    # Bill Conti Oscars musical director

    # Music Industry (7)
    "131.txt": 7,    # Franz Ferdinand government help (duplicate check)

    # Celebrity Culture (9)
    "280.txt": 9,    # Bill Conti Oscars — actually music industry crossover
}

# Fix duplicate key — 280.txt belongs in Music Industry
noise_remap["280.txt"] = 7

# Build initial DataFrame 
final_df = pd.DataFrame({
    'filename':         unique_files,
    'category':         'Entertainment',
    'sub_category_id':  topics,
    'sub_category':     [topic_labels.get(t, 'Unassigned') for t in topics],
    'confidence_score': [round(float(p), 4) if p > 0 else 0.0 for p in probs],
    'preview':          unique_Entertainment
})

# Apply all remapping in one pass 
def consolidate_and_remap(row):
    """
    Three jobs in one pass — applied in priority order:
    1. Noise articles (-1) → assign correct label via noise_remap
    2. Misclassified articles → correct via manual_remap
    3. Look up final label from topic_labels
    """
    sid = row['sub_category_id']

    # Step 1 — handle noise
    if sid == -1:
        sid = noise_remap.get(row['filename'], -1)

    # Step 2 — fix known misclassified articles
    if row['filename'] in manual_remap:
        sid = manual_remap[row['filename']]

    return sid, topic_labels.get(sid, 'Unassigned')

final_df[['sub_category_id', 'sub_category']] = final_df.apply(
    consolidate_and_remap, axis=1, result_type='expand'
)

# Summary table 
print("\nBBC ENTERTAINMENT — SUB-CATEGORY SUMMARY (fully labelled)")
print("=" * 65)

summary = final_df.groupby(['sub_category_id', 'sub_category']).agg(
    article_count=('filename', 'count'),
    avg_confidence=('confidence_score', 'mean')
).reset_index()

summary['avg_confidence'] = summary['avg_confidence'].round(4)
summary = summary.sort_values('sub_category_id')

for _, row in summary.iterrows():
    print(f"\n  [{int(row['sub_category_id'])}] {row['sub_category']}")
    print(f"       Articles: {row['article_count']} | Avg Confidence: {row['avg_confidence']}")

print(f"\n{'=' * 65}")
print(f"Total articles    : {len(final_df)}")
print(f"Sub-categories    : {final_df['sub_category_id'].nunique()}")
print(f"Unassigned (noise): {(final_df['sub_category_id'] == -1).sum()}")

# Full DataFrame display
print("\nFull DataFrame:")
display(final_df[['filename', 'category', 'sub_category', 'confidence_score', 'preview']])